In [1]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Mon Apr 27 10:56:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8       

In [2]:
%%writefile vector_ops.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <iostream>
#include <cstdlib>
#include <ctime>

using namespace std;

__global__ void VecMulKernel(float *a, float *b, float *c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) c[i] = a[i] * b[i];
}

__global__ void VecAddKernel(float *a, float *b, float *c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) c[i] = a[i] + b[i];
}

void vec_mul_cuda(float *a, float *b, float *c, int n) {
    int size = n * sizeof(float);
    float *a_gpu, *b_gpu, *c_gpu;
    cudaMalloc(&a_gpu, size);
    cudaMalloc(&b_gpu, size);
    cudaMalloc(&c_gpu, size);
    cudaMemcpy(a_gpu, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(b_gpu, b, size, cudaMemcpyHostToDevice);
    int threads = 512;
    int blocks = (n + threads - 1) / threads;
    VecMulKernel<<<blocks, threads>>>(a_gpu, b_gpu, c_gpu, n);
    cudaDeviceSynchronize();
    cudaMemcpy(c, c_gpu, size, cudaMemcpyDeviceToHost);
    cudaFree(a_gpu); cudaFree(b_gpu); cudaFree(c_gpu);
}

void vec_add_cuda(float *a, float *b, float *c, int n) {
    int size = n * sizeof(float);
    float *a_gpu, *b_gpu, *c_gpu;
    cudaMalloc(&a_gpu, size);
    cudaMalloc(&b_gpu, size);
    cudaMalloc(&c_gpu, size);
    cudaMemcpy(a_gpu, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(b_gpu, b, size, cudaMemcpyHostToDevice);
    int threads = 512;
    int blocks = (n + threads - 1) / threads;
    VecAddKernel<<<blocks, threads>>>(a_gpu, b_gpu, c_gpu, n);
    cudaDeviceSynchronize();
    cudaMemcpy(c, c_gpu, size, cudaMemcpyDeviceToHost);
    cudaFree(a_gpu); cudaFree(b_gpu); cudaFree(c_gpu);
}

int main() {
    srand(time(NULL));
    const int N = 1024;
    float a[N], b[N], c_mul[N], c_add[N];
    for (int i = 0; i < N; i++) {
        a[i] = rand() % 10;
        b[i] = rand() % 10;
    }
    vec_mul_cuda(a, b, c_mul, N);
    vec_add_cuda(a, b, c_add, N);
    cout << "Умножение (первые 20): ";
    for (int i = 0; i < 20; i++) cout << c_mul[i] << " ";
    cout << endl;
    cout << "Сложение (первые 20): ";
    for (int i = 0; i < 20; i++) cout << c_add[i] << " ";
    cout << endl;
    return 0;
}

Writing vector_ops.cu


In [3]:
!nvcc -o vector_ops vector_ops.cu
!./vector_ops

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Умножение (первые 20): 12 15 8 16 4 0 0 48 81 45 21 12 0 6 16 2 0 8 15 9 
Сложение (первые 20): 8 8 9 10 5 4 2 14 18 14 10 8 7 5 10 3 7 9 8 10 


In [4]:
%%writefile gpu_info.cu
#include <iostream>
#include <cuda.h>
#include <cuda_runtime.h>

using namespace std;

int main() {
    int device_count;
    cudaGetDeviceCount(&device_count);
    cout << "CUDA devices found: " << device_count << endl;
    for (int i = 0; i < device_count; i++) {
        cudaDeviceProp dp;
        cudaGetDeviceProperties(&dp, i);
        cout << "\n=== Device " << i << " ===" << endl;
        cout << "Name: " << dp.name << endl;
        cout << "Total global memory: " << dp.totalGlobalMem / (1024*1024) << " MB" << endl;
        cout << "Shared memory per block: " << dp.sharedMemPerBlock / 1024 << " KB" << endl;
        cout << "Constant memory: " << dp.totalConstMem / 1024 << " KB" << endl;
        cout << "Registers per block: " << dp.regsPerBlock << endl;
        cout << "Warp size: " << dp.warpSize << endl;
        cout << "Max threads per block: " << dp.maxThreadsPerBlock << endl;
        cout << "Compute capability: " << dp.major << "." << dp.minor << endl;
        cout << "Multiprocessors: " << dp.multiProcessorCount << endl;
        cout << "Core clock rate: " << dp.clockRate / 1000 << " MHz" << endl;
        cout << "Memory clock rate: " << dp.memoryClockRate / 1000 << " MHz" << endl;
        cout << "Memory bus width: " << dp.memoryBusWidth << " bits" << endl;
        cout << "L2 cache size: " << dp.l2CacheSize / 1024 << " KB" << endl;
    }
    return 0;
}

Writing gpu_info.cu


In [5]:
!nvcc -o gpu_info gpu_info.cu
!./gpu_info

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA devices found: 1

=== Device 0 ===
Name: Tesla T4
Total global memory: 14912 MB
Shared memory per block: 48 KB
Constant memory: 64 KB
Registers per block: 65536
Warp size: 32
Max threads per block: 1024
Compute capability: 7.5
Multiprocessors: 40
Core clock rate: 1590 MHz
Memory clock rate: 5001 MHz
Memory bus width: 256 bits
L2 cache size: 4096 KB


In [6]:
%%writefile bandwidth.cu
#include <iostream>
#include <cuda.h>
#include <cuda_runtime.h>

using namespace std;

const int SIZE = 100 * 1024 * 1024;

int main() {
    char *h_src = (char*)malloc(SIZE);
    char *h_dst = (char*)malloc(SIZE);
    char *h_pinned = NULL;
    char *d_src = NULL;
    char *d_dst = NULL;

    for (int i = 0; i < SIZE; i++) h_src[i] = rand() % 256;

    cudaMalloc(&d_src, SIZE);
    cudaMalloc(&d_dst, SIZE);
    cudaMallocHost(&h_pinned, SIZE);
    memcpy(h_pinned, h_src, SIZE);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float elapsed;

    cudaEventRecord(start);
    memcpy(h_dst, h_src, SIZE);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&elapsed, start, stop);
    cout << "Host -> Host: " << SIZE / (1024.0*1024*1024) / (elapsed/1000) << " GB/s" << endl;

    cudaEventRecord(start);
    cudaMemcpy(d_src, h_src, SIZE, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&elapsed, start, stop);
    cout << "Host -> Device: " << SIZE / (1024.0*1024*1024) / (elapsed/1000) << " GB/s" << endl;

    cudaEventRecord(start);
    cudaMemcpy(h_dst, d_src, SIZE, cudaMemcpyDeviceToHost);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&elapsed, start, stop);
    cout << "Device -> Host: " << SIZE / (1024.0*1024*1024) / (elapsed/1000) << " GB/s" << endl;

    cudaEventRecord(start);
    cudaMemcpy(d_dst, d_src, SIZE, cudaMemcpyDeviceToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&elapsed, start, stop);
    cout << "Device -> Device: " << SIZE / (1024.0*1024*1024) / (elapsed/1000) << " GB/s" << endl;

    cudaEventRecord(start);
    cudaMemcpy(d_src, h_pinned, SIZE, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&elapsed, start, stop);
    cout << "Pinned Host -> Device: " << SIZE / (1024.0*1024*1024) / (elapsed/1000) << " GB/s" << endl;

    free(h_src); free(h_dst);
    cudaFreeHost(h_pinned);
    cudaFree(d_src); cudaFree(d_dst);
    return 0;
}

Writing bandwidth.cu


In [7]:
!nvcc -o bandwidth bandwidth.cu
!./bandwidth

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Host -> Host: 1.52029 GB/s
Host -> Device: 4.28339 GB/s
Device -> Host: 4.04046 GB/s
Device -> Device: 109.468 GB/s
Pinned Host -> Device: 11.2614 GB/s


In [8]:
%%writefile matrix_mul.cu
#include <iostream>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>
#include <stdlib.h>
#include <time.h>

using namespace std;

const int TILE_SIZE = 16;

// Обычное умножение на GPU
__global__ void matMulClassic(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

// Умножение с общей памятью (блочное)
__global__ void matMulShared(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE_SIZE][TILE_SIZE];
    __shared__ float Bs[TILE_SIZE][TILE_SIZE];

    int bx = blockIdx.x, by = blockIdx.y;
    int tx = threadIdx.x, ty = threadIdx.y;

    int row = by * TILE_SIZE + ty;
    int col = bx * TILE_SIZE + tx;

    float sum = 0.0f;

    for (int k = 0; k < (N + TILE_SIZE - 1) / TILE_SIZE; k++) {
        if (row < N && k * TILE_SIZE + tx < N) {
            As[ty][tx] = A[row * N + k * TILE_SIZE + tx];
        } else {
            As[ty][tx] = 0.0f;
        }
        if (col < N && k * TILE_SIZE + ty < N) {
            Bs[ty][tx] = B[(k * TILE_SIZE + ty) * N + col];
        } else {
            Bs[ty][tx] = 0.0f;
        }
        __syncthreads();

        for (int i = 0; i < TILE_SIZE; i++) {
            sum += As[ty][i] * Bs[i][tx];
        }
        __syncthreads();
    }

    if (row < N && col < N) {
        C[row * N + col] = sum;
    }
}

// CPU версия для проверки
void matMulCPU(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * N + j];
            }
            C[i * N + j] = sum;
        }
    }
}

// Сравнение матриц
bool compareMatrices(const float* C1, const float* C2, int N, float eps = 0.001f) {
    for (int i = 0; i < N * N; i++) {
        if (fabs(C1[i] - C2[i]) > eps) {
            cout << "Mismatch at " << i << ": " << C1[i] << " vs " << C2[i] << endl;
            return false;
        }
    }
    return true;
}

int main() {
    srand(time(NULL));
    const int N = 512;
    const int size = N * N * sizeof(float);

    float *h_A = new float[N*N];
    float *h_B = new float[N*N];
    float *h_C_CPU = new float[N*N];
    float *h_C_GPU_classic = new float[N*N];
    float *h_C_GPU_shared = new float[N*N];

    for (int i = 0; i < N*N; i++) {
        h_A[i] = rand() % 10;
        h_B[i] = rand() % 10;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    double t1 = clock();
    matMulCPU(h_A, h_B, h_C_CPU, N);
    double t_cpu = (clock() - t1) / CLOCKS_PER_SEC * 1000;

    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (N + 15) / 16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);
    matMulClassic<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float t_classic;
    cudaEventElapsedTime(&t_classic, start, stop);
    cudaMemcpy(h_C_GPU_classic, d_C, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(start);
    matMulShared<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float t_shared;
    cudaEventElapsedTime(&t_shared, start, stop);
    cudaMemcpy(h_C_GPU_shared, d_C, size, cudaMemcpyDeviceToHost);

    cout << "N=" << N << endl;
    cout << "CPU: " << t_cpu << " ms" << endl;
    cout << "GPU classic: " << t_classic << " ms (x" << t_cpu / t_classic << ")" << endl;
    cout << "GPU shared: " << t_shared << " ms (x" << t_cpu / t_shared << ")" << endl;
    cout << "Classic vs CPU: " << (compareMatrices(h_C_CPU, h_C_GPU_classic, N) ? "OK" : "FAIL") << endl;
    cout << "Shared vs CPU: " << (compareMatrices(h_C_CPU, h_C_GPU_shared, N) ? "OK" : "FAIL") << endl;

    delete[] h_A; delete[] h_B; delete[] h_C_CPU;
    delete[] h_C_GPU_classic; delete[] h_C_GPU_shared;
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [9]:
!nvcc -o matrix_mul matrix_mul.cu
!./matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
N=512
CPU: 686.491 ms
GPU classic: 38.2834 ms (x17.9318)
GPU shared: 0.484512 ms (x1416.87)
Classic vs CPU: OK
Shared vs CPU: OK
